# ALGAE_DATA.csv 비트리 모델 입력 데이터 생성

이 노트북은 `src/data/team-raw/ALGAE_DATA.csv`에서 **비트리 모델용 스케일링 데이터셋**을 만드는 과정을 정리한 실행 문서입니다.

비트리 모델 예시:

- Logistic Regression
- Linear/Ridge/Lasso Regression
- SVM
- KNN
- Neural Network
- PCA / clustering

이런 모델들은 feature scale에 민감하므로, 수질/조류/댐 운영 컬럼까지 로그 변환 또는 스케일링합니다.

최종 저장 파일:

```text
src/data/processed/model_input/non_tree_scaled/algae_non_tree_scaled_station_expanded.csv
```

## 1. 설계 요약

입력 데이터 `ALGAE_DATA.csv`는 이미 아래 데이터가 병합된 station-expanded 데이터입니다.

```text
수질/조류/댐 운영 데이터 + station별 기상 데이터
```

구조상 같은 `date + loc_encoded` 수질 행이 station `604, 643, 648, 888`별로 4번 반복됩니다.

비트리 모델용 변환 정책:

| 컬럼군 | 처리 |
| --- | --- |
| `station`, `loc_encoded` | 원값 보존 + one-hot encoding |
| 조류 세포수, 탁도, Chl-a | `log10(x + 1)` 후 `RobustScaler` |
| 강우, 유입, 방류, 정체 지표 | `RobustScaler` |
| 수온, pH, DO, 수위, 저수량 등 제한 범위 변수 | `MinMaxScaler` |
| 종별 비율, 고온일수, 저풍속일수 | 그대로 유지 |
| `next_log_cells` | 회귀 target, 스케일링 금지 |
| `target_alert_next` | 분류 target, `next_log_cells >= log10(1000 + 1)` |

중요: scaler는 `train` 행에만 fit하고 `valid` 행에는 transform만 적용합니다.

In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, RobustScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = SRC_DIR / "data"
INPUT_PATH = DATA_DIR / "team-raw" / "ALGAE_DATA.csv"
OUTPUT_DIR = DATA_DIR / "processed" / "model_input" / "non_tree_scaled"
OUTPUT_PATH = OUTPUT_DIR / "algae_non_tree_scaled_station_expanded.csv"
METADATA_PATH = OUTPUT_DIR / "preprocessing_metadata.json"

INPUT_PATH, OUTPUT_PATH

In [ ]:
df = pd.read_csv(INPUT_PATH)
print(df.shape)
df.head()

In [ ]:
print("결측치 총합:", int(df.isna().sum().sum()))
print("완전 중복 행:", int(df.duplicated().sum()))
print("날짜 범위:", df["date"].min(), "~", df["date"].max())
print("station별 행 수")
print(df["station"].value_counts().sort_index())
print("loc_encoded별 행 수")
print(df["loc_encoded"].value_counts().sort_index())

## 2. Target과 시간 기준 split 생성

- 회귀 target: `next_log_cells`
- 분류 target: `target_alert_next`
- 관심 기준: 유해남조류 세포수 1,000 이상
- `next_log_cells`는 `log10(cells + 1)` 스케일이므로 threshold도 `log10(1000 + 1)`로 변환합니다.

검증 성능 과대평가를 막기 위해 날짜 기준으로 뒤쪽 20%를 validation으로 둡니다.

In [ ]:
DATE_COLUMN = "date"
TARGET_REGRESSION = "next_log_cells"
TARGET_CURRENT_LOG = "log_target"
TARGET_ALERT_NEXT = "target_alert_next"
ALERT_CELL_THRESHOLD = 1000

work = df.copy()
work[DATE_COLUMN] = pd.to_datetime(work[DATE_COLUMN])
threshold_log10 = np.log10(ALERT_CELL_THRESHOLD + 1)
work[TARGET_ALERT_NEXT] = (work[TARGET_REGRESSION] >= threshold_log10).astype(int)

unique_dates = pd.Series(work[DATE_COLUMN].sort_values().unique())
valid_start = unique_dates.iloc[int(len(unique_dates) * 0.8)]
work["split"] = np.where(work[DATE_COLUMN] >= valid_start, "valid", "train")
work = work.sort_values([DATE_COLUMN, "loc_encoded", "station"]).reset_index(drop=True)

print("valid_start:", valid_start)
print(work["split"].value_counts())
print("target_alert_next rate:", work[TARGET_ALERT_NEXT].mean())

## 3. 컬럼 그룹 정의

전처리팀의 기상 컬럼 스케일링 의도를 유지하면서, 비트리 모델에 민감한 수질/조류/댐 운영 컬럼도 함께 변환합니다.

In [ ]:
WEATHER_ROBUST_COLS = [
    "daily_rain",
    "rain_3d_sum",
    "rain_7d_sum_y",
    "rain_14d_sum",
]

WEATHER_MINMAX_COLS = [
    "avg_temp",
    "min_temp",
    "max_temp",
    "avg_wind",
    "air_temp_7d_mean",
    "wind_7d_mean",
    "sunshine",
    "solar_rad",
    "cloud_cover",
    "sunshine_7d_sum",
    "solar_7d_sum",
    "cloud_7d_mean",
    "max_wind_gust",
    "sin_season",
    "cos_season",
]

WATER_QUALITY_MINMAX_COLS = [
    "water_temp",
    "pH",
    "DO",
    "transparency",
    "acc_temp_7d",
    "TSI_Chla",
    "TSI_SD",
    "water_level",
    "storage_vol",
    "storage_rate",
]

WATER_HYDRO_ROBUST_COLS = [
    "level_change_7d",
    "inflow",
    "outflow",
    "rainfall",
    "rain_7d_sum_x",
    "inflow_7d_sum",
    "outflow_7d_sum",
    "residence_proxy",
    "nutrient_stagnation_index",
]

LOG_THEN_ROBUST_COLS = [
    "turbidity",
    "Chl_a",
    "cyano_cells",
    "Microcystis",
    "Anabaena",
    "Oscillatoria",
    "Aphanizomenon",
]

PASS_THROUGH_FEATURE_COLS = [
    "microcystis_ratio",
    "anabaena_ratio",
    "oscillatoria_ratio",
    "aphanizomenon_ratio",
    "hot_days_30c_7d",
    "low_wind_days_2ms_7d",
]

TARGET_COLUMNS = [TARGET_REGRESSION, TARGET_CURRENT_LOG, "alert_encoded", TARGET_ALERT_NEXT]

In [ ]:
def fit_transform_column_group(data, columns, scaler, train_mask, suffix=""):
    existing = [col for col in columns if col in data.columns]
    output = pd.DataFrame(index=data.index)
    fitted = scaler.fit(data.loc[train_mask, existing])
    values = fitted.transform(data[existing])
    output_columns = [f"{col}{suffix}" for col in existing]
    output[output_columns] = values
    return output

train_mask = work["split"].eq("train")
parts = [
    work[[DATE_COLUMN, "split", "station", "loc_encoded"]].copy(),
    pd.get_dummies(work["station"], prefix="station", dtype=int),
    pd.get_dummies(work["loc_encoded"], prefix="loc", dtype=int),
]

parts.append(fit_transform_column_group(work, WEATHER_ROBUST_COLS, RobustScaler(), train_mask, "_robust"))
parts.append(fit_transform_column_group(work, WEATHER_MINMAX_COLS, MinMaxScaler(), train_mask, "_minmax"))
parts.append(fit_transform_column_group(work, WATER_QUALITY_MINMAX_COLS, MinMaxScaler(), train_mask, "_minmax"))
parts.append(fit_transform_column_group(work, WATER_HYDRO_ROBUST_COLS, RobustScaler(), train_mask, "_robust"))

logged = np.log10(work[LOG_THEN_ROBUST_COLS] + 1)
logged.columns = [f"{col}_log10p1" for col in LOG_THEN_ROBUST_COLS]
parts.append(fit_transform_column_group(logged, list(logged.columns), RobustScaler(), train_mask, "_robust"))

parts.append(work[PASS_THROUGH_FEATURE_COLS].copy())
parts.append(work[TARGET_COLUMNS].copy())

non_tree_df = pd.concat(parts, axis=1)
non_tree_df[DATE_COLUMN] = pd.to_datetime(non_tree_df[DATE_COLUMN]).dt.strftime("%Y-%m-%d")
print(non_tree_df.shape)
non_tree_df.head()

## 4. 저장 및 검증

생성된 파일은 `src/data/processed/model_input/non_tree_scaled/` 아래에 저장됩니다.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
non_tree_df.to_csv(OUTPUT_PATH, index=False)

metadata = {
    "source": str(INPUT_PATH.relative_to(PROJECT_ROOT)),
    "output": str(OUTPUT_PATH.relative_to(PROJECT_ROOT)),
    "fit_policy": "Scalers are fit on rows where split == 'train' only, then applied to valid rows.",
    "target_regression": TARGET_REGRESSION,
    "target_alert_next": TARGET_ALERT_NEXT,
    "alert_cell_threshold": ALERT_CELL_THRESHOLD,
    "valid_start": str(pd.Timestamp(valid_start).date()),
    "row_count": int(len(non_tree_df)),
    "column_count": int(non_tree_df.shape[1]),
}
METADATA_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2))

print("saved:", OUTPUT_PATH)
print("metadata:", METADATA_PATH)

In [ ]:
check = pd.read_csv(OUTPUT_PATH)
print("shape:", check.shape)
print("missing:", int(check.isna().sum().sum()))
print("duplicates:", int(check.duplicated().sum()))
print("split:")
print(check["split"].value_counts())
print("target rate:", check[TARGET_ALERT_NEXT].mean())
check.head()

## 5. 사용 시 주의

- 이 파일은 station-expanded 구조입니다. 같은 `date + loc_encoded` 수질 행이 station 4개별로 반복됩니다.
- 모델 검증은 반드시 날짜 기준으로 나누어야 합니다.
- `next_log_cells`, `target_alert_next`는 target이므로 feature에 넣으면 안 됩니다.
- `log_target`은 현재 시점 세포수 로그값입니다. 현재 상태 feature로 쓸 수 있지만, 회귀 target인 `next_log_cells`와 혼동하지 않아야 합니다.